# T2.L4 Demo — CPM, PERT та Monte Carlo

Мета: пройти шлях від мережевої моделі до оцінювання deadline probability.

In [ ]:
from pathlib import Path
import sys, pandas as pd, matplotlib.pyplot as plt
ROOT = Path.cwd().parents[0] if Path.cwd().name == "notebooks" else Path.cwd()
if (ROOT / "src").exists() is False:
    ROOT = Path("..")
sys.path.insert(0, str(ROOT / "src"))
from model import cpm_schedule, apply_delay, pert_parameters, monte_carlo_project, deadline_probability
tasks = pd.read_csv(ROOT / "data" / "tasks.csv")
pert = pd.read_csv(ROOT / "data" / "pert.csv")
tasks

## 1. Baseline CPM

In [ ]:
duration, schedule, critical_path = cpm_schedule(tasks)
print("Project duration:", duration)
print("Critical path:", " -> ".join(critical_path))
schedule

## 2. Delay scenarios

In [ ]:
for task, delay in [("C",3),("D",3)]:
    changed = apply_delay(tasks, task, delay)
    d, s, p = cpm_schedule(changed)
    print(f"{task}+{delay}: duration={d}, path={' -> '.join(p)}")

## 3. PERT parameters

In [ ]:
params = pert_parameters(pert)
params

## 4. Monte Carlo

In [ ]:
mc, freq = monte_carlo_project(tasks, pert, n=3000, seed=2026)
print("Mean:", mc.project_duration.mean())
print("P50/P80/P90:", mc.project_duration.quantile([.5,.8,.9]).to_dict())
print("P(T<=19):", deadline_probability(mc,19))
freq.head()

In [ ]:
plt.figure(figsize=(8,5))
plt.hist(mc["project_duration"], bins=30)
plt.axvline(19, linestyle="--", label="deadline 19")
plt.xlabel("Project duration")
plt.ylabel("Simulation count")
plt.title("Distribution of project duration")
plt.legend()
plt.show()

## Інтерпретація

- CPM дає структуру критичності за фіксованих durations.
- Slack пояснює, чому не кожна затримка переносить project finish.
- PERT/Monte Carlo переводять питання з одного строку до розподілу можливих строків.
- Імовірність завершення до deadline залежить від прийнятих оцінок і розподілів.